# Data

In [2]:

import os, sys, yaml
print("cwd =", os.getcwd())
print("sys.path[0] =", sys.path[0])

from src.data import (
    load_data, clean_data,
    filter_min_group_size, get_group_counts,
    make_splits
)


with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
    data_cfg = cfg["data"]
    feature_cfg = cfg["feature"]
    models_cfg = cfg["models"]
    cv_cfg = cfg["cv"]
    run_cfg = cfg.get("run", {})
    models_to_run = run_cfg.get("models", list(models_cfg.keys()))
    missing_cfg = cfg.get("missing", None)
    predict_cfg = cfg["predict"]



cwd = /Users/shimengdai/Dropbox/Michigan State/Papers/MCRNI/prediction
sys.path[0] = /opt/anaconda3/envs/CMSE/lib/python312.zip


In [5]:
df = load_data(data_cfg)
df2 = clean_data(df, data_cfg)

print(df2.head())
print(df2["School_ID"].dtype)


  School_ID  dropout   AP  attendance  suspension  misbehavior  math_t_score  \
0    1011.0        0  0.0         1.0         1.0          0.0         52.11   
1    1011.0        0  0.0         1.0         1.0          0.0         57.65   
2    1011.0        0  0.0         1.0         1.0          0.0         66.44   
3    1011.0        0  0.0         NaN         1.0          NaN         44.68   
4    1011.0        0  0.0         1.0         1.0          0.0         40.57   

   reading_t_score  college_enroll  gpa  extracurricular  stem_cred  \
0            59.53             NaN  2.0              0.0        NaN   
1            56.70             1.0  4.0              2.0        NaN   
2            64.46             1.0  4.0              2.0        0.0   
3            48.69             0.0  4.0             14.0        0.0   
4            33.53             2.0  4.0              0.0        NaN   

   stem_courses  stem_gpa  hard_stem_career  soft_stem_career  
0           5.0      1.00   

In [6]:
min_size = 10
school_col = data_cfg["school_id_col"]

df_filtered = filter_min_group_size(
    df=df2,
    group_col=school_col,
    min_size=min_size
)

counts_after = df_filtered[school_col].value_counts()
counts_after



School_ID
4401.0    46
4271.0    46
4341.0    41
2911.0    35
4101.0    31
          ..
1461.0    10
1452.0    10
3102.0    10
4431.0    10
9997.0    10
Name: count, Length: 665, dtype: int64

In [7]:
n_schools_before = df2[school_col].nunique()
n_schools_after  = df_filtered[school_col].nunique()

print("Schools before filtering:", n_schools_before)
print("Schools after filtering :", n_schools_after)
print("Schools removed         :", n_schools_before - n_schools_after)


Schools before filtering: 751
Schools after filtering : 665
Schools removed         : 86


In [8]:
print(len(df2))
print(len(df_filtered))

16197
11813


In [9]:
get_group_counts(df2, data_cfg["school_id_col"])

,School_ID,n_samples
0,4272.0,1
1,4512.0,1
2,4343.0,2
3,3051.0,2
4,4531.0,2
...,...,...
746,4101.0,31
747,2911.0,35
748,4341.0,41
749,4271.0,46


In [10]:

get_group_counts(df_filtered, data_cfg["school_id_col"])

,School_ID,n_samples
0,9997.0,10
1,1452.0,10
2,1461.0,10
3,1591.0,10
4,1612.0,10
...,...,...
660,4101.0,31
661,2911.0,35
662,4341.0,41
663,4271.0,46


In [12]:
school_col = "School_ID"
df_train, df_test = make_splits(df_filtered, school_col, test_size=0.2, seed=42)

# quick sanity check
set(df_train[school_col]).isdisjoint(set(df_test[school_col]))


True

In [13]:
print(len(df_train))
print(len(df_test))

9369
2444


# Features

In [14]:
from src.features import get_feature_columns, make_xy

feature_cfg  = {
"exclude_cols": ["stem_cred"]
}

feature_cols = get_feature_columns(
    df_train,
    data_cfg=data_cfg,
    feature_cfg=feature_cfg,
)

feature_cols 

['AP',
 'attendance',
 'suspension',
 'misbehavior',
 'math_t_score',
 'reading_t_score',
 'college_enroll',
 'gpa',
 'extracurricular',
 'stem_courses',
 'stem_gpa',
 'hard_stem_career',
 'soft_stem_career']

In [15]:

X, y, groups = make_xy(
    df_train,
    data_cfg=data_cfg,
    feature_cols=feature_cols
)

print("n_features:", len(feature_cols))
print("X shape:", X.shape)
print("y shape:", y.shape, "unique:", y.nunique())
print("groups shape:", groups.shape, "n_groups:", groups.nunique())


n_features: 13
X shape: (9369, 13)
y shape: (9369,) unique: 2
groups shape: (9369,) n_groups: 532


In [17]:

print(len(groups))
print(groups.nunique())
print(groups.value_counts().describe())
set(groups.unique()) == set(df_train[data_cfg["school_id_col"]].unique())


9369
532
count    532.000000
mean      17.610902
std        5.037049
min       10.000000
25%       14.000000
50%       17.000000
75%       21.000000
max       46.000000
Name: count, dtype: float64


True

# Models

In [18]:

from src.models import build_model
from sklearn.pipeline import Pipeline

model_name = "logistic_regression"
seed = 42

model_cfg = models_cfg[model_name]          
est = build_model(model_name, model_cfg, seed=seed)

print("Model name:", model_name)
print("Type      :", type(est))
print("Estimator :", est)

if isinstance(est, Pipeline):
    print("Pipeline steps:", est.named_steps.keys())


Model name: logistic_regression
Type      : <class 'sklearn.pipeline.Pipeline'>
Estimator : Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(max_iter=1000, random_state=42,
                                    solver='liblinear'))])
Pipeline steps: dict_keys(['scaler', 'model'])


In [19]:
print(models_cfg["random_forest"].keys())
print(models_cfg["random_forest"].get("param_grid"))


dict_keys(['params', 'param_grid'])
{'n_estimators': [200, 300], 'min_samples_leaf': [1, 5], 'max_depth': [None, 10]}


In [20]:
from src.train import run_grid_search
model_name = "logistic_regression" #logistic_regression
model_cfg = models_cfg[model_name]
search = run_grid_search(
    X=X,
    y=y,
    groups=groups,
    model_name=model_name,
    model_cfg=model_cfg,
    cv_cfg=cv_cfg,
    seed=data_cfg["seed"],
    missing_cfg =  missing_cfg,
)

print("Best CV score :", search.best_score_)
print("Best params   :", search.best_params_)
print("Best estimator:")
print(search.best_estimator_)


Best CV score : 0.8525487517378355
Best params   : {'model__C': 0.1, 'model__penalty': 'l2'}
Best estimator:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=0.1, max_iter=1000, random_state=57,
                                    solver='liblinear'))])


In [21]:
from src.train import run_grid_search
model_name = "svm" #logistic_regression
model_cfg = models_cfg[model_name]
search = run_grid_search(
    X=X,
    y=y,
    groups=groups,
    model_name=model_name,
    model_cfg=model_cfg,
    cv_cfg=cv_cfg,
    seed=data_cfg["seed"],
    missing_cfg =  missing_cfg,
)

print("Best CV score :", search.best_score_)
print("Best params   :", search.best_params_)
print("Best estimator:")
print(search.best_estimator_)

Best CV score : 0.8333629645708303
Best params   : {'model__C': 0.1, 'model__gamma': 'scale'}
Best estimator:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 SVC(C=0.1, class_weight='balanced', probability=True,
                     random_state=57))])


In [21]:
from src.train import run_grid_search
model_name = "random_forest" #logistic_regression
model_cfg = models_cfg[model_name]
search = run_grid_search(
    X=X,
    y=y,
    groups=groups,
    model_name=model_name,
    model_cfg=model_cfg,
    cv_cfg=cv_cfg,
    seed=data_cfg["seed"],
    missing_cfg =  missing_cfg,
)

print("Best CV score :", search.best_score_)
print("Best params   :", search.best_params_)
print("Best estimator:")
print(search.best_estimator_)

Best CV score : 0.8702012274452191
Best params   : {'model__max_depth': None, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}
Best estimator:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('model',
                 RandomForestClassifier(min_samples_leaf=5, n_estimators=300,
                                        n_jobs=-1, random_state=57))])


In [7]:
#import sys
from pathlib import Path
import pandas as pd

#PROJECT_ROOT = Path.cwd()   # prediction/
#sys.path.append(str(PROJECT_ROOT))

from src.predict import predict_one
from src.io import load_model
from src.features import get_feature_columns, make_xy


input_path = Path(predict_cfg["input_path"])
df_pred = pd.read_csv(input_path)
print(df_pred.shape)

models_dir = Path(predict_cfg["models_dir"])
first_model_cfg = predict_cfg["models"][0]
model_path = models_dir / first_model_cfg["file"]

model = load_model(model_path)
model

(2272, 16)


,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [8]:
import pandas as pd

df_test = pd.read_csv(predict_cfg["input_path"])
df_test.shape


feature_cols = get_feature_columns(
    df_test,
    data_cfg=data_cfg,
    feature_cfg=feature_cfg,
    )

X_test, y_test, g_test = make_xy(
        df_test,
        data_cfg=data_cfg,
        feature_cols=feature_cols,
    )

print(
        "X_test      :",
        X_test.shape,
        "y_test :",
        y_test.shape,
        "groups:",
        g_test.nunique(),
    )

X_test      : (2272, 13) y_test : (2272,) groups: 133


In [ ]:
y_pred_proba = predict_one(model, X_test, pred_type="proba")

type(y_pred_proba), y_pred_proba.shape

print(y_pred_proba)

[0.02730854 0.00321293 0.00534155 ... 0.00440255 0.00426276 0.00534718]


In [4]:
import pandas as pd 

prediction_wide = pd.read_csv("/Users/shimengdai/Dropbox/Michigan State/Papers/MCRNI/prediction/data/predictions/predictions_wide.csv")

y_test = prediction_wide["dropout"]
y_pred_proba = prediction_wide["pred_rf_proba"]
y_test.mean()


np.float64(0.027288732394366196)

In [5]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

f1 = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = np.argmax(f1)

best_threshold = thresholds[best_idx]
print("Best threshold:", best_threshold)


Best threshold: 0.1518130801586379
